In [ ]:
# === Mount Google Drive and install dependencies ===
from google.colab import drive
drive.mount("/content/drive")
!pip install -r /content/drive/MyDrive/iris/requirements.txt -q

# 06 — C3: Detection Comparison

**Purpose:** Train and evaluate all three detection approaches on the same
train/test split, then compare precision, recall, F1, and ROC-AUC.

**The three approaches (Design Document §4.4):**
1. Classical baseline: TF-IDF + Logistic Regression (text features)
2. Raw activation baseline: Logistic Regression on 1280-dim residual stream
3. SAE feature detector: Logistic Regression on 10240-dim SAE features

**Key question:** Does the SAE decomposition produce features that are more
discriminative than raw activations? If SAE features outperform raw activations,
that's evidence the decomposition found useful structure.

**Prerequisites:**
- `checkpoints/sae_d10240_lambda1e-04.pt`
- `checkpoints/j2_activations.npz`
- `checkpoints/sensitivity_scores.npy` (from notebook 05)
- `data/processed/iris_dataset_balanced.json`

*Nathan Cheung (ncheung3@my.yorku.ca) | York University | CSSD 2221 | Winter 2026*

In [ ]:
import sys, os
IN_COLAB = 'google.colab' in sys.modules
PROJECT_ROOT = '/content/drive/MyDrive/iris' if IN_COLAB else os.path.abspath(os.path.join(os.getcwd(), '..'))
sys.path.insert(0, PROJECT_ROOT)
os.chdir(PROJECT_ROOT)

from src.utils.helpers import set_seed, get_device
set_seed(42)
device = get_device()

In [ ]:
# === Load dataset and split ===
import torch
import numpy as np
import json
from src.data.dataset import IrisDataset
from src.sae.architecture import SparseAutoencoder
from src.analysis.features import compute_feature_activations

dataset = IrisDataset.load('data/processed/iris_dataset_balanced.json')

# Split into train/test (70/30 for simplicity — C3 uses held-out test set)
# We use a simple split here rather than train/val/test because we're not
# tuning hyperparameters — just comparing fixed approaches.
from sklearn.model_selection import train_test_split

indices = list(range(len(dataset)))
train_idx, test_idx = train_test_split(
    indices, test_size=0.3, stratify=dataset.labels, random_state=42
)

train_texts = [dataset.texts[i] for i in train_idx]
test_texts = [dataset.texts[i] for i in test_idx]
train_labels = np.array([dataset.labels[i] for i in train_idx])
test_labels = np.array([dataset.labels[i] for i in test_idx])

print(f'Train: {len(train_texts)} ({train_labels.sum()} injection)')
print(f'Test:  {len(test_texts)} ({test_labels.sum()} injection)')

In [ ]:
# === Load activations and SAE features ===
import json

data = np.load('checkpoints/j2_activations.npz')

with open('results/metrics/j2_evaluation.json') as f:
    j2_metrics = json.load(f)
TARGET_LAYER = j2_metrics['train_layer']
all_activations = data[f'layer_{TARGET_LAYER}']

train_activations = all_activations[train_idx]
test_activations = all_activations[test_idx]

# Load SAE and compute feature activations
checkpoint = torch.load('checkpoints/sae_d10240_lambda1e-04.pt', map_location=device)
config = checkpoint['config']
sae = SparseAutoencoder(d_input=config['d_input'], expansion_factor=config['expansion_factor'],
                        sparsity_coeff=config.get('sparsity_coeff', 1e-4))
sae.load_state_dict(checkpoint['model_state_dict'])
sae = sae.to(device)
sae.eval()

# Compute SAE features for train and test sets
all_features = compute_feature_activations(sae, all_activations, device=device)
train_features = all_features[train_idx]
test_features = all_features[test_idx]

print(f'Layer: {TARGET_LAYER}')
print(f'Train activations: {train_activations.shape}')
print(f'Test activations:  {test_activations.shape}')
print(f'Train SAE features: {train_features.shape}')
print(f'Test SAE features:  {test_features.shape}')

In [ ]:
# === Load sensitivity scores for top-K ablation ===
from pathlib import Path

sensitivity_path = Path('checkpoints/sensitivity_scores.npy')
if sensitivity_path.exists():
    sensitivity = np.load(sensitivity_path)
    print(f'Loaded sensitivity scores: {sensitivity.shape}')
else:
    # Compute on the fly if notebook 05 wasn't run first
    from src.analysis.features import compute_sensitivity_scores
    sensitivity = compute_sensitivity_scores(all_features, np.array(dataset.labels))

In [ ]:
# === Run the three-way comparison ===
from src.analysis.detection import run_detection_comparison, plot_roc_comparison, plot_metrics_comparison

comparison_results, predictions_dict = run_detection_comparison(
    train_texts=train_texts,
    train_labels=train_labels,
    test_texts=test_texts,
    test_labels=test_labels,
    train_activations=train_activations,
    test_activations=test_activations,
    train_features=train_features,
    test_features=test_features,
    sensitivity_scores=sensitivity,
    top_k_values=[10, 50, 100],  # A2 ablation: how few features suffice?
)

In [ ]:
# === ROC curves ===
plot_roc_comparison(
    comparison_results, test_labels, predictions_dict,
    save_path='results/figures/c3_roc_comparison.png'
)

In [ ]:
# === Metrics comparison bar chart ===
plot_metrics_comparison(
    comparison_results,
    save_path='results/figures/c3_metrics_comparison.png'
)

In [ ]:
# === Save C3 results ===
c3_results = {
    'experiment': 'C3',
    'train_size': len(train_texts),
    'test_size': len(test_texts),
    'layer': TARGET_LAYER,
    'results': comparison_results,
}
with open('results/metrics/c3_detection_comparison.json', 'w') as f:
    json.dump(c3_results, f, indent=2, default=lambda x: float(x))
print('Saved to results/metrics/c3_detection_comparison.json')

## C3 Summary

Compared three detection approaches on the same train/test split:
1. TF-IDF + LogReg (text-level features)
2. Raw activations + LogReg (1280-dim residual stream)
3. SAE features + LogReg (10240-dim sparse features)

Also ran A2 ablation: detection with only the top 10, 50, 100 features.

**Key findings:** Compare the F1 and AUC columns in the table above.
If SAE features outperform raw activations, the decomposition adds value.